[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/quantitative_imaging.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Quanitative Image Reconstruction - T2 Mapping using Mono-Exponential Model

For the reconstruction of quantitative maps we will have to select the different signal models.

In [ ]:
import numpy as np
import torch
from pathlib import Path
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian
from mrpro.operators import DictionaryMatchOp
from mrpro.operators.models import MonoExponentialDecay
from cmap import Colormap

## Select the raw data files

In [ ]:
raw_files = list(
    Path(
        "/Users/kolbit01/Documents/Data/Joseba/Quantitative knee images i3M/Subject4-LeftKnee-T2map"
    ).rglob("*.h5")
)
image_folder = "/Users/kolbit01/Documents/Data/Joseba/Quantitative knee images i3M/Subject4-LeftKnee-T2map/Dicom"
te_prep = None


# raw_files = ['/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Raw/Left Knee/meas_MID00128_FID21426_t2map_trufi_Left.mrd',]
# image_folder = '/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/Dicom/Left Knee/T2Map Trufi/t2map_trufi_Left_MOCO_T2_map'
# te_prep =[ 0.0, 0.025, 0.055, 0.0, 0.025, 0.06]

vmax = 0.15

## Load Dicom

In [ ]:
t2_map_dicom = IData.from_dicom_folder(image_folder)

# Dicom saves T2 values in ms, we use seconds here.
t2_map_dicom.data *= 1e-3

## Reconstruct images

Data obtained after different inversion times can be available in a single file or as individual files. 
These can be simply read it one by one and then concatenated into a single kdata object.

In [ ]:
kdata = [
    KData.from_file(fname, trajectory=KTrajectoryCartesian()) for fname in raw_files
]
kdata = kdata[0].concatenate(*kdata[1:], dim=0)

# for multi-slice acquisition: resort the data into correct order of slices and contrasts
if (nslices := kdata.header.acq_info.idx.slice.unique().numel()) > 1:
    sort_idx = torch.argsort(kdata.header.acq_info.position.x.squeeze(), stable=True)
    kdata = kdata[sort_idx].rearrange(
        "(slice contrast) ... -> contrast slice ...", slice=nslices
    )

# We have to calculate the coil maps from one of the contrasts which ideally has high signal for all tissue types
csm = CsmData.from_kdata_inati(kdata[0], downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
idata = recon(kdata)

## Create signal model for inversion recovery sequence and calculate quantitative maps

We will select the correct signal model and then create a dictionary mapping operator, similarly what is used in MR Fingerprinting. 
We can then simply apply the dictionary mapping operator to the reconstructed images and obtain the quantitative maps

In [ ]:
dictionary = DictionaryMatchOp(
    MonoExponentialDecay(decay_time=te_prep if te_prep else idata.header.te),
    index_of_scaling_parameter=0,
)
dictionary.append(torch.tensor(1.0), torch.linspace(0.01, 0.6, 1000)[None, :])
m0_match, t2_match = dictionary(idata.data)
t2_map = IData(t2_match, header=idata.header[0])

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from einops import rearrange
import nibabel as nib

from nibabel.orientations import (
    io_orientation,
    axcodes2ornt,
    ornt_transform,
    apply_orientation,
)


def show_image(qmap: IData, cmap, vmax: float) -> None:
    """Show the qualitative images."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))

    for cax in ax.flatten():
        cax.set_axis_off()

    def orient_images(idata: IData) -> np.array:
        # Ensure the slices are in the correct order
        if idata.shape[0] > 1:
            sort_idx = torch.argsort(idata.header.position.x.squeeze())
            idata = idata[sort_idx]
        orientation = idata.header.orientation.as_matrix().squeeze()
        affine_zyx = torch.cat(
            [
                torch.tensor([[1.0, 0.0, 0.0, 0.0]]),
                torch.cat([torch.zeros((3, 1)), orientation], 1),
            ],
            0,
        )

        data = rearrange(idata.data, "... other z y x-> x y z 1 other (...)")
        img_nii = nib.Nifti1Image(
            data.squeeze().abs().numpy(force=True),
            affine_zyx.flip([0, 1]),
            dtype=np.float32,
        )
        # Target orientation (RAS)
        ras_ornt = axcodes2ornt(("R", "S", "A"))
        transform = ornt_transform(io_orientation(img_nii.affine), ras_ornt)
        ras_data = apply_orientation(img_nii.get_fdata(), transform)
        return ras_data

    def plot_multi_slice_image(ax, img, colorbar_label, cmap, vmax):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        idim = img.squeeze().shape[0]
        for cax, slice_idx in zip(
            ax, [int(idim // 2 - idim // 6), idim // 2, int(idim // 2 + idim // 6)]
        ):
            im = cax.imshow(
                np.squeeze(img[slice_idx, :, :]),
                cmap=cmap,
                vmin=0,
                vmax=vmax,
                origin="lower",
            )
            fig.colorbar(im, ax=cax, label=colorbar_label)

    plot_multi_slice_image(ax, orient_images(qmap), "T1 (s)", cmap, vmax)

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(t2_map, Colormap("navia").to_mpl(), vmax)
show_image(t2_map_dicom, Colormap("navia").to_mpl(), vmax)